# Optimize clustered synthetic data parameters

This notebook searches for clustered (non-spatial) synthetic data parameters such that:

- IID TOST **passes** at Δ=1
- Cluster-robust TOST **fails** at Δ=1

This demonstrates within-building dependence in the paired difference that IID inference misses.

In [1]:
from pyTOST.data_gen.optimize_cluster_params import search, save_best
from pyTOST.data_gen.params_io import load_params

ALPHA = 0.05
MARGIN = 1.0
SEED = 123


ImportError: cannot import name 'search' from 'pyTOST.data_gen.optimize_cluster_params' (/Users/dhetting/src/pyTOST/pyTOST/data_gen/optimize_cluster_params.py)

In [ ]:
best = search(n_trials=800, alpha=ALPHA, margin=MARGIN, seed=SEED, rng_seed=SEED, verbose_every=50)

In [ ]:
print(f'IID: {best.eq_iid}\nCluster: {best.eq_cluster}')

In [ ]:
print(f'IID: {best.ci_iid}\nCluster: {best.ci_cluster}')

In [ ]:
# Save best params
out_json = 'best_params_cluster.json'
save_best(best, out_json)
print('Wrote', out_json)

In [ ]:
# Example: load params from disk and reuse
params = load_params(out_json)
params


In [ ]:
# Generate a dataset using loaded params
from pyTOST.data_gen.synthetic_tost_data import generate_cluster_groups
import pandas as pd

params = params
df_long, meta = generate_cluster_groups(**params)
# Build diff df
A = df_long[df_long['arm']=='A'][['sample_id','group_id','x','y_sp']].copy()
wide = df_long.pivot(index='sample_id', columns='arm', values='y').reset_index()
df_diff = A.merge(wide, on='sample_id')
df_diff['diff'] = df_diff['B'] - df_diff['A']
df_diff = df_diff.rename(columns={'y_sp':'y'})
df_diff.head()

In [ ]:
# Run IID and cluster engines
from pyTOST.engines import iid_tost, cluster_tost

iid = iid_tost.IIDTOST(y='diff')
clu = cluster_tost.ClusterTOST(y='diff', cluster='group_id')

res_iid = iid.fit(df_diff, alpha=ALPHA, margins=[MARGIN])
res_clu = clu.fit(df_diff, alpha=ALPHA, margins=[MARGIN])
res_iid, res_clu
